
# Week 13 LAB 13 Advanced Demo: Source-Grounded GenAI with Gemini API

This notebook is an **optional advanced version** of the Week 13 source-grounded AI lab. It shows how to:

1. Download the EPA 2024-2027 Climate Adaptation Plan PDF.
2. Extract selected pages as source text.
3. Ask Gemini to answer using only the provided source text.
4. Save the AI response for use in a short reflection.

**Important:** This notebook is not training or fine-tuning Gemini. It is giving Gemini a source excerpt in the prompt and asking it to answer only from that source. This is a simple source-grounded/RAG-like classroom demonstration.

Default source: U.S. Environmental Protection Agency, 2024-2027 Climate Adaptation Plan.



## Step 0: API key setup

To run the Gemini API cells, you need a Gemini API key. In Colab:

1. Open the **Secrets** panel on the left side.
2. Add a secret named `GEMINI_API_KEY`.
3. Paste your Gemini API key as the value.
4. Turn notebook access on for that secret.

The rest of the notebook can be used to download and inspect the source text even without an API key.


In [ ]:

# Install required packages
!pip install -q -U google-genai pypdf requests


In [ ]:

from pathlib import Path
import requests
from pypdf import PdfReader
import textwrap
import os

EPA_PDF_URL = "https://www.epa.gov/system/files/documents/2024-06/epas-2024-2027-climate-adaptation-plan-508-compliant.pdf"
PDF_PATH = Path("epa_2024_2027_climate_adaptation_plan.pdf")

if not PDF_PATH.exists():
    r = requests.get(EPA_PDF_URL, timeout=60)
    r.raise_for_status()
    PDF_PATH.write_bytes(r.content)

print(f"Downloaded: {PDF_PATH}")
print(f"File size: {PDF_PATH.stat().st_size/1_000_000:.2f} MB")


In [ ]:

# Extract selected page ranges from the PDF.
# Page numbers below are PDF page numbers shown to students, starting at 1.
# You can adjust these ranges if your instructor wants a different focus.

reader = PdfReader(str(PDF_PATH))
print("Number of PDF pages:", len(reader.pages))

# Recommended focus sections for this lab:
# 1-3: title/policy statement and contents
# 5-8: agency profile/summary statement
# 9-18: risk assessment
# 61-65: progress measures
selected_pages = list(range(1, 4)) + list(range(5, 19)) + list(range(61, 66))

source_chunks = []
for page_num in selected_pages:
    idx = page_num - 1
    if idx < len(reader.pages):
        text = reader.pages[idx].extract_text() or ""
        text = " ".join(text.split())
        source_chunks.append(f"[PDF page {page_num}]\n{text}")

source_text = "\n\n".join(source_chunks)
print("Characters extracted:", len(source_text))
print(source_text[:2000])



## Step 1: Create a source-grounded prompt

The prompt below asks the model to use **only** the source excerpt. This is important for a source-grounded exercise.

Students should look for:

- Does the answer cite or point to the source pages?
- Does the answer avoid adding unsupported information?
- What is missing because the source excerpt is incomplete?
- Where is human review needed?


In [ ]:

question = """
Based only on the source excerpt from the EPA 2024-2027 Climate Adaptation Plan, answer the following:

1. What are the main climate hazards or risks discussed?
2. How does the plan discuss environmental justice or affected communities?
3. What adaptation actions or implementation areas are described?
4. What information is missing or unclear if someone wanted to use this document for local community planning?
5. What should a human reviewer check before using this answer in policy, planning, or communication?
"""

prompt = f"""
You are a source-grounded AI assistant for an introductory environmental management course.
Use ONLY the source excerpt provided below. Do not use outside knowledge.
If the source excerpt does not contain enough information, say what is missing.
For each main claim, include a short source note using the PDF page number shown in brackets.
Keep the answer concise and suitable for undergraduate students.

SOURCE EXCERPT:
{source_text[:60000]}

QUESTION:
{question}
"""

print(prompt[:3000])



## Step 2: Call Gemini API

This cell tries the current Gemini SDK interaction method first, and falls back to the older `models.generate_content` style if needed. API interfaces can change, so this helper is written to be a little more robust for classroom use.


In [ ]:

from google import genai

# Get API key from Colab Secrets or environment variable
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("No GEMINI_API_KEY found. Add it in Colab Secrets or set it as an environment variable.")

client = genai.Client(api_key=api_key)

# Use a broadly available flash model name if your account supports it.
# If your API returns a model error, replace this with a model listed in your Gemini API console.
MODEL_NAME = "gemini-3.5-flash"

def ask_gemini(prompt, model=MODEL_NAME):
    # Newer interactions API style
    try:
        interaction = client.interactions.create(model=model, input=prompt)
        if hasattr(interaction, "output_text"):
            return interaction.output_text
        return str(interaction)
    except Exception as e1:
        print("Interactions API call failed; trying models.generate_content fallback...")
        print("First error:", repr(e1))
        # Older generate_content style
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            if hasattr(response, "text"):
                return response.text
            return str(response)
        except Exception as e2:
            print("Fallback call also failed.")
            raise e2

answer = ask_gemini(prompt)
print(answer)


In [ ]:

# Save response to a text file that students can download or copy into their worksheet.
output_path = Path("week13_source_grounded_ai_response.txt")
output_path.write_text(answer, encoding="utf-8")
print(f"Saved: {output_path}")



## Step 3: Student reflection questions

After reviewing the answer, respond to these questions in the submission worksheet:

1. Which parts of the answer were clearly supported by the source excerpt?
2. Were any parts vague, unsupported, or hard to verify?
3. Did the answer address environmental justice or affected communities?
4. What information was missing because of the source document or source excerpt?
5. What should a human reviewer check before using this answer in planning, policy, or public communication?


In [ ]:

# Optional: Try your own question.
student_question = """
Based only on the EPA Climate Adaptation Plan source excerpt, what are two accountability or progress-measurement ideas mentioned in the plan?
"""

student_prompt = f"""
Use ONLY the source excerpt below. If the answer is not present, say that it is not found in the source excerpt.
Include short page-number notes for claims.

SOURCE EXCERPT:
{source_text[:60000]}

QUESTION:
{student_question}
"""

# Uncomment to run:
# student_answer = ask_gemini(student_prompt)
# print(student_answer)
